In [ ]:
#COGNITIVE TEST IMPAIREMENT THRESHOLD
"""This code is used to evaluate the z-score level that should classify a test as representing 'impaired' performance. 
We chose to classify impaired cognition as  ≥ 2 tests impaired in at least one assessment instance (same-instance criterion).
We tested 4 thresholds and calculated how each determined the resulting prevalence of cognitive impairment and the 
diagnostic performance in dementia. """


import pandas as pd
import numpy as np
import os

data_path = "cv_results_all_tests_merged_with_groups.csv"
out_dir   = "IC_threshold"
os.makedirs(out_dir, exist_ok=True)

THRESHOLDS    = [1.0, 1.5, 2.0, 2.5]
REVERSE_TESTS = ["PMat", "RT", "TMA", "TMB"]

In [ ]:
#LOAD DATA
"""
Loads the annotated merged dataset produced by 05e_group_definition.ipynb.
A participant-level frame is built (one row per eid) for group-level prevalence and sens/spec.
"""

df = pd.read_csv(data_path)

participants = df.drop_duplicates(subset="eid")[["eid", "dem_all"]].copy()
participants["dem_all"] = participants["dem_all"].fillna(0).astype(int)

n_total     = len(participants)
n_dementia  = int(participants["dem_all"].sum())

print(f"Total participants: {n_total:,}")
print(f"Ever-dementia:      {n_dementia:,} ({100*n_dementia/n_total:.2f}%)")
print(f"Tests in dataset:   {sorted(df['test_name'].unique())}")

In [ ]:
#IC AT THRESHOLD
"""
Re-computes impairment from z-scores at an arbitrary SD threshold and applies the
same-instance IC rule (≥ 2 impaired tests in at least one assessment instance).
PMem uses its pre-calculated binary impairment (no z-score available) at every threshold.
Returns the set of eids classified as IC.
"""

def classify_IC(df, threshold):
    pmem_mask  = df["test_name"] == "PMem"
    rev_mask   = df["test_name"].isin(REVERSE_TESTS) & ~pmem_mask
    other_mask = ~df["test_name"].isin(REVERSE_TESTS) & ~pmem_mask

    imp = pd.Series(0, index=df.index, dtype=int)
    imp[pmem_mask]  = df.loc[pmem_mask,  "impaired"].fillna(0).astype(int)
    imp[rev_mask]   = (df.loc[rev_mask,   "z_score"] >  threshold).astype(int)
    imp[other_mask] = (df.loc[other_mask, "z_score"] < -threshold).astype(int)

    tmp = df[["eid", "instance"]].copy()
    tmp["imp"] = imp.values

    inst_counts = tmp.groupby(["eid", "instance"])["imp"].sum().reset_index()
    IC_eids    = set(inst_counts.loc[inst_counts["imp"] >= 2, "eid"].unique())
    return IC_eids

In [ ]:
#THRESHOLD SWEEP
"""
Classifies each participant as IC / not-IC at every threshold and stores the flag
as IC_<threshold> in the participant-level frame.
"""

for t in THRESHOLDS:
    IC_eids = classify_IC(df, t)
    participants[f"IC_{t}"] = participants["eid"].isin(IC_eids).astype(int)
    print(f"Threshold {t} SD:  {len(IC_eids):,} IC ({100*len(IC_eids)/n_total:.2f}%)")

In [ ]:
#PREVALENCE TABLE
"""
Cohort prevalence of IC at each threshold, with the dementia overlap.
"""

rows = []
for t in THRESHOLDS:
    col       = f"IC_{t}"
    n_IC     = int(participants[col].sum())
    n_IC_dem = int(((participants[col]==1) & (participants["dem_all"]==1)).sum())
    rows.append({
        "Threshold (SD)":            t,
        "N IC":                     n_IC,
        "Prevalence IC (%)":        round(100 * n_IC / n_total, 2),
        "N IC + dementia":          n_IC_dem,
        "N IC, no dementia":        n_IC - n_IC_dem,
        "% of IC who get dementia": round(100 * n_IC_dem / n_IC, 2) if n_IC > 0 else 0,
    })

prev_table = pd.DataFrame(rows)
prev_table.to_csv(f"{out_dir}/IC_prevalence_by_threshold.csv", index=False)
prev_table

In [ ]:
#SENSITIVITY / SPECIFICITY
"""
Performance of the IC classifier for eventual dementia diagnosis at each threshold.
  Sensitivity = TP / (TP + FN)   proportion of dementia cases flagged as IC
  Specificity = TN / (TN + FP)   proportion of non-dementia cases NOT flagged as IC
  PPV         = TP / (TP + FP)   proportion of IC who go on to dementia
  NPV         = TN / (TN + FN)   proportion of non-IC who never develop dementia
"""

rows = []
for t in THRESHOLDS:
    col = f"IC_{t}"
    IC = participants[col] == 1
    dem = participants["dem_all"] == 1

    tp = int((IC & dem).sum())
    fp = int((IC & ~dem).sum())
    tn = int((~IC & ~dem).sum())
    fn = int((~IC & dem).sum())

    sens = tp / (tp + fn) if (tp + fn) > 0 else np.nan
    spec = tn / (tn + fp) if (tn + fp) > 0 else np.nan
    ppv  = tp / (tp + fp) if (tp + fp) > 0 else np.nan
    npv  = tn / (tn + fn) if (tn + fn) > 0 else np.nan

    rows.append({
        "Threshold (SD)": t,
        "TP": tp, "FP": fp, "TN": tn, "FN": fn,
        "Sensitivity":   round(sens, 4),
        "Specificity":   round(spec, 4),
        "PPV":           round(ppv,  4),
        "NPV":           round(npv,  4),
    })

sens_spec = pd.DataFrame(rows)
sens_spec.to_csv(f"{out_dir}/IC_sensitivity_specificity.csv", index=False)
sens_spec

In [ ]:
#THRESHOLD SUMMARY
"""
Combined trade-off table for threshold selection: prevalence × sens × spec × PPV × NPV.
Choose the threshold that best balances clinical plausibility, IC prevalence,
and predictive performance for dementia conversion.
"""

prev = prev_table[["Threshold (SD)", "N IC", "Prevalence IC (%)"]]
perf = sens_spec[["Threshold (SD)", "Sensitivity", "Specificity", "PPV", "NPV"]]
summary = prev.merge(perf, on="Threshold (SD)")

summary.to_csv(f"{out_dir}/IC_threshold_summary.csv", index=False)

print("="*80)
print("IC THRESHOLD SELECTION SUMMARY")
print("="*80)
print(summary.to_string(index=False))
print("="*80)
summary

In [ ]:
#SAVE IC CASE LISTS
"""
Persists the full per-instance test data for the IC cohort at each threshold,
preserving time_to_dementia_diagnosis for downstream Cox PH analysis.
"""

for t in THRESHOLDS:
    col   = f"IC_{t}"
    eids  = set(participants.loc[participants[col]==1, "eid"])
    sub   = df[df["eid"].isin(eids)].sort_values(["eid", "instance", "test_name"])
    path  = f"{out_dir}/IC_{str(t).replace('.','p')}sd.csv"
    sub.to_csv(path, index=False)
    print(f"Threshold {t} SD: {len(eids):,} participants  |  {len(sub):,} rows → {path}")

# Also save the participant-level flags for joining onto other analyses
participants.to_csv(f"{out_dir}/participant_IC_flags.csv", index=False)
print(f"\nParticipant flags → {out_dir}/participant_IC_flags.csv")